In [2]:
# import modules
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
#from scipy.stats import vonmises_fisher

# import emcee for autocorrelation
import emcee
# load the result object
from holopy.core.holopy_object import HoloPyObject, FullLoader
from holopy.core.utils import ensure_array, dict_without
import yaml
import importlib
# load using h5py
import h5py as h5

import holopy as hp
from holopy.core.process import normalize, bg_correct, center_find, subimage
from holopy.scattering import Sphere, Spheres, calc_holo
from holopy.inference import prior, ExactModel, CmaStrategy, EmceeStrategy, AlphaModel, NmpfitStrategy
from holopy.inference import model

In [3]:
#needed to make display work properly (there are other options as well if this fails)
%matplotlib tk

In [5]:
# path to directory
DIRECTORYPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/'

## Load fit

In [6]:
# Load one of Caroline's fits
Caroline_load_path = '/Volumes/manoharan_lab/cmartin/Fits/depletion_holography/09-21-21/depletion/0.0875/03/mcmcdimer_frame0_mcmc.h5'
Caroline_fit = hp.load(Caroline_load_path)

In [7]:
SAVEPATH = DIRECTORYPATH+ 'compare_Caroline_fit/depletion_holography_09-21-21_depletion_0.0875_03_mcmcdimer_frame0'
print(SAVEPATH)
results5 = Caroline_fit

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/compare_Caroline_fit/depletion_holography_09-21-21_depletion_0.0875_03_mcmcdimer_frame0


## Work on futher data processing (ie drop pre-burn in data, drop non-converged fits, and decimate remaining data so its independent)

In [159]:
# samples using wrong order of indexing as an example of what not to do
samples = results5.samples
print(samples[:,999][0])
print(means)

IndexError: index 999 is out of bounds for axis 1 with size 50

In [160]:
print(Caroline_fit.samples)

<xarray.DataArray 'samples' (walker: 1000, chain: 50, parameter: 11)>
array([[[ 1.5848412 ,  0.67486494, 58.52464038, ...,  1.60176338,
          0.64461416,  0.64784015],
        [ 1.58483884,  0.67486986, 58.49950336, ...,  1.60177418,
          0.64461127,  0.64595734],
        [ 1.58487128,  0.67480212, 58.49289661, ...,  1.6017802 ,
          0.64465044,  0.65408324],
        ...,
        [ 1.58486572,  0.67493602, 58.52920279, ...,  1.60177027,
          0.64469937,  0.64365069],
        [ 1.58484842,  0.67493615, 58.53172686, ...,  1.60181956,
          0.64459465,  0.67596474],
        [ 1.58484805,  0.67496107, 58.51084154, ...,  1.6018423 ,
          0.6446162 ,  0.64927638]],

       [[ 1.58483955,  0.67488006, 58.5223107 , ...,  1.60176539,
          0.6446409 ,  0.64890306],
        [ 1.58482755,  0.67485196, 58.49233114, ...,  1.60177788,
          0.64458614,  0.64098541],
        [ 1.58487128,  0.67480212, 58.49289661, ...,  1.6017802 ,
          0.64465044,  0.65408324

In [163]:
print(samples[:,0].sel(parameter='phi'))

<xarray.DataArray 'samples' (chain: 50)>
array([6.32904312, 6.32727058, 6.3082288 , 6.31900603, 6.33040046,
       6.32838131, 6.32657128, 6.3280753 , 6.32680198, 6.33708578,
       6.32972014, 6.32424436, 6.33195153, 6.33289885, 6.32859743,
       6.32294746, 6.32260451, 6.32497189, 6.32907306, 6.32821182,
       6.31889525, 6.3267322 , 6.3311869 , 6.32687177, 6.33028549,
       6.31510257, 6.3188733 , 6.33047821, 6.32111792, 6.33061369,
       6.3361368 , 6.32445699, 6.33005593, 6.3275473 , 6.32426272,
       6.33034321, 6.33471398, 6.33171986, 6.32167335, 6.32940588,
       6.31913187, 6.33541714, 6.33387036, 6.32629558, 6.32168691,
       6.33257709, 6.33046834, 6.31197749, 6.3232634 , 6.31718385])
Coordinates:
    parameter  <U3 'phi'
Dimensions without coordinates: chain
Attributes:
    acceptance_fraction:  0.41444


In [165]:
# select gap parameter values for first walker (all chains)
gaps_example = samples[:,1].sel(parameter='gap')
print(gaps_example)

<xarray.DataArray 'samples' (walker: 1000)>
array([0.09807602, 0.09781792, 0.09781792, 0.09769245, 0.09769245,
       0.09769245, 0.09800655, 0.09831327, 0.09831327, 0.09831327,
       0.09833753, 0.09833753, 0.09857736, 0.09853451, 0.0985321 ,
       0.0985321 , 0.0985321 , 0.0985321 , 0.0985321 , 0.0985321 ,
       0.0985321 , 0.0985321 , 0.0985321 , 0.0985321 , 0.098504  ,
       0.09839383, 0.09839383, 0.09822146, 0.09810331, 0.09810331,
       0.09810331, 0.09810331, 0.09810331, 0.09796654, 0.09798941,
       0.09811715, 0.09837315, 0.09837315, 0.09837329, 0.09837329,
       0.09829401, 0.09829401, 0.09829401, 0.09829401, 0.09830439,
       0.09830439, 0.09830439, 0.09830326, 0.09830326, 0.09830326,
       0.09830326, 0.09830326, 0.09830326, 0.09830326, 0.09832028,
       0.09832028, 0.09832028, 0.09832028, 0.09832028, 0.09832028,
       0.09832028, 0.09832557, 0.09832557, 0.09832557, 0.0983083 ,
       0.0983083 , 0.0983083 , 0.0983083 , 0.09833148, 0.09828588,
       0.09828588,

In [166]:
plt.figure()
plt.plot(gaps_example)

### Drop pre-burn in (right now ad hoc but come back to make more systematic)

In [167]:
# plot pre-burn in
plt.figure()
plt.title('Pre burn-in logprob of fit')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(11):
    plt.plot(results5.lnprobs[:,i])
plt.savefig(SAVEPATH + '/pre_burn_in_lnprob.png')

In [502]:
# looking at the fits for all the different walkers it's clear that several don't converge well
plt.figure()
plt.title('Post burn-in logprob showing bad fits')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(30):
    plt.plot(burnt_results5.lnprobs[i])
plt.savefig(SAVEPATH + '/many_lnprobs_to_show_bad_fits.png')

### Visualize Data Traces

In [169]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
#plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[:,i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/few_theta_fits.png')

In [171]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
#plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[:,i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/many_phi_fits.png')

In [173]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[:,i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/few_gap_fits.png')

In [175]:
# look at some traces of r1
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[:,i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/many_r1_fits.png')

In [177]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[:,i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/few_r2_fits.png')

In [179]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
#plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[:,i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/many_n1_fits.png')

In [182]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
#plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[:,i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/few_n2_fits.png')

In [184]:
# look at some traces of x_g
plt.figure()
plt.title("Central x position (um) fit")
plt.ylabel('x (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Xg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(results5.samples[:,i].sel(parameter='x_g'))
plt.savefig(SAVEPATH + '/few_xg_fits.png')

In [186]:
# look at some traces of y_g
plt.figure()
plt.title("Central y position (um) fit")
plt.ylabel('y (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Yg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(results5.samples[:,i].sel(parameter='y_g'))
plt.savefig(SAVEPATH + '/many_yg_fits.png')

In [188]:
# look at some traces of z_g
plt.figure()
plt.title("Central z position (um) fit")
plt.ylabel('z (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Zg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(results5.samples[:,i].sel(parameter='z_g'))
plt.savefig(SAVEPATH + '/few_zg_fits.png')

### Drop bad convergence (ie low lnprob) walkers. Later attempt to fix these fits instead.

In [97]:
# look at distribution of final lnprob values across walkers
new_sample_length = len(burnt_results5.lnprobs[0])
plt.figure()
plt.plot(burnt_results5.lnprobs[:,(new_sample_length-1)])
plt.savefig(SAVEPATH + '/final_lnprob_values_across_walkers')
maxlnprob = max(burnt_results5.lnprobs[:,(new_sample_length-1)])
converged_value = maxlnprob - 0.02*maxlnprob
print(converged_value)

<xarray.DataArray 'lnprobs' ()>
array(-20763.85675686)


In [25]:
# The bad fits are the swapped angles fits (visible in the reorganized pandas dataset)
# -> is there a good way to correct these or should I just drop them?
# Start by implementing a cutoff in lnprobs to drop them and then can work on fixing later
samples = burnt_results5.samples
converged_samples_nan = xr.DataArray()
# 0 doesn't work as a cutoff universally, example, 3000 chain fit 1 needs 15000 as cutoff
bad_fit_index = []
good_fit_index = []
for i in range(len(samples)):
    new_sample_length = len(burnt_results5.lnprobs[i])
    if burnt_results5.lnprobs[i][new_sample_length-1] > converged_value:
        converged_samples_nan = xr.concat([converged_samples_nan,samples[i]],'walker')
        # .append() isn't quite what we want, try to use xarray methods
        good_fit_index.append(i)
    else:
        bad_fit_index.append(i)
converged_samples = converged_samples_nan[1:]
print(converged_samples)
print(len(converged_samples))
print(bad_fit_index)

<xarray.DataArray (walker: 50, chain: 900, parameter: 11)>
array([[[1.58482057, 0.67495495, 4.99984056, ..., 1.60183524,
         0.64467746, 0.99792262],
        [1.58482057, 0.67495495, 4.99984056, ..., 1.60183524,
         0.64467746, 0.99792262],
        [1.58482057, 0.67495495, 4.99984056, ..., 1.60183524,
         0.64467746, 0.99792262],
        ...,
        [1.58449521, 0.67562803, 4.99983405, ..., 1.60154275,
         0.64446082, 1.00000199],
        [1.58449521, 0.67562803, 4.99983405, ..., 1.60154275,
         0.64446082, 1.00000199],
        [1.58449521, 0.67562803, 4.99983405, ..., 1.60154275,
         0.64446082, 1.00000199]],

       [[1.58486099, 0.67499093, 5.00020593, ..., 1.6018123 ,
         0.64457582, 1.0002579 ],
        [1.58486434, 0.67495533, 4.99990368, ..., 1.60179627,
         0.64460339, 0.99937249],
        [1.58486204, 0.67494157, 4.99984262, ..., 1.60178945,
         0.64462652, 0.99980008],
...
        [1.58484093, 0.67463005, 4.99973664, ..., 1.602067

In [ ]:
# could also do this with ds.drop_sel(space=["IN", "IL"]) where we use walker = [drop indexes]

In [49]:
# plot converged samples
# need to modify this for each fit you want to use it for
# converged_id = [0,1,2,3,4,5,6,8,9]
plt.figure()
for id in good_fit_index:
    plt.plot(burnt_results5.lnprobs[id])

NameError: name 'converged_id' is not defined

#### Visualize good vs. bad fit parameter traces

In [437]:
# plot bad fits to see parrellels between them
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/30089_bad_theta_fits.png')

In [438]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/30089_bad_phi_fits.png')

In [439]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(results5.samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/30089_bad_gap_fits.png')

In [622]:
# plot good fits to see parrellels between them
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/30089_good_theta_fits.png')

In [ ]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/30089_good_phi_fits.png')

In [ ]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(results5.samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/30089_good_gap_fits.png')

#### Visualize fits with low starting phi values (near boundary)

In [717]:
# look at how many starting phi look like the starting phi that lead to these bad fits
low_start_index = []
for i in range(50):
    if (samples[i,0].sel(parameter='phi') > np.pi) and (samples[i,0].sel(parameter='phi') < 2*np.pi):
        low_start_index.append(i)
print(low_start_index)

[0, 2, 5, 6, 9, 13, 18, 19, 20, 21, 22, 26, 27, 28, 38, 42, 44, 49]


In [613]:
# for this run 16 is bad since it swaps phi to around pi
low_start_index.remove(16)
print(low_start_index)

[0, 1, 2, 3, 5, 6, 7, 8, 9, 18, 22, 23, 25, 26, 27, 31, 36, 38, 40, 42, 43, 44, 45, 47]


In [718]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/low_starting_phi_lnprobs.png')

In [615]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id][100:])
plt.savefig(SAVEPATH + '/low_starting_phi_lnprobs_drop_first_220.png')

In [719]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/low_starting_phi_theta_fit.png')

In [720]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit.png')

In [64]:
# look at some traces of phi that are never off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi') > 4):
        plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit_not_off_by_pi.png')

In [121]:
# look at the end of some traces of phi that are not off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi')[-50:-1] > 4):
        plt.plot(samples[i].sel(parameter='phi')[-50:-1])

In [65]:
# plot starting phi positions for low initial conditions that aren't off by mod pi
plt.figure()
plot_varible = []
for id in low_start_index:
    if samples[id,0].sel(parameter='phi') > 4:
        plot_varible.append(samples[id,0].sel(parameter='phi'))
plt.plot(plot_varible)
plt.savefig(SAVEPATH + '/low_starting_phi_plot_of_starting_phi_not_off_by_pi.png')

In [721]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/low_starting_phi_gap_fit.png')

In [722]:
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_r1_fit.png')

In [723]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_r2_fit.png')

In [724]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_n1_fit.png')

In [725]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_n2_fit.png')

In [ ]:
# look at the fits that start with lnprob like the bad fits but then jump to good fits
# these are subset of low_start index and so already convered

#### Visualize fits with high starting phi values (near boundary)

In [726]:
# look at how many starting phi look like the starting phi that lead to bad fits
high_start_index = []
for i in range(50):
    if (samples[i,0].sel(parameter='phi') < 1):
        high_start_index.append(i)
print(high_start_index)

[30, 34, 35, 37, 41, 43, 45, 46, 48]


In [727]:
plt.figure()
for id in high_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/high_starting_phi_lnprobs.png')

In [728]:
plt.figure()
for id in high_start_index:
    plt.plot(burnt_results5.lnprobs[id][100:])
plt.savefig(SAVEPATH + '/high_starting_phi_lnprobs_drop_first_350.png')

In [729]:
# look at some traces of theta with high starting phi
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/high_starting_phi_theta_fit.png')

In [732]:
# look at some traces of phi with high starting phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/high_starting_phi_phi_fit.png')

In [731]:
# look at some traces of gap with high starting phi
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/high_starting_phi_gap_fit.png')

In [733]:
# look at some traces of r_1 with high starting phi
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/high_starting_phi_r1_fit.png')

In [734]:
# look at some traces of r2 with high starting phi
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/high_starting_phi_r2_fit.png')

In [735]:
# look at some traces of n1 with high starting phi
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/high_starting_phi_n1_fit.png')

In [736]:
# look at some traces of n2 with high starting phi
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/high_starting_phi_n2_fit.png')

#### Visualize fits with low starting theta values

In [442]:
# look at how many starting phi look like the starting phi that lead to these bad fits
low_start_index = []
for i in range(30):
    if (results5.samples[i,0].sel(parameter='theta') > np.pi) and (results5.samples[i,0].sel(parameter='phi') < 2*np.pi):
        low_start_index.append(i)
print(low_start_index)

[2, 5, 9, 12, 14, 15, 16, 19, 20, 21, 23, 24]


In [443]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/boundary_starting_theta_lnprobs.png')

In [446]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_theta_fit.png')

In [447]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_phi_fit.png')

In [449]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_gap_fit.png')

In [450]:
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_r1_fit.png')

In [451]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_r2_fit.png')

In [452]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_n1_fit.png')

In [453]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_n2_fit.png')

### Decimate data so just keep independent fits

In [8]:
# for weird load that swapped walkers and chains bypass earlier steps
converged_samples = results5.samples

In [9]:
# look at autocorrelation to see how long it is before lose memory so can decimate into independent samples
series = pd.Series(converged_samples[:,1].sel(parameter='gap'))
autocorr_as_function_of_time = []
for i in range(len(series)):
    autocorr = series.autocorr(lag=i)
    autocorr_as_function_of_time.append(autocorr)
plt.figure()
plt.plot(autocorr_as_function_of_time)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


In [167]:
# look at what corresponding trace looks like
plt.figure()
plt.plot(converged_samples[:,1].sel(parameter='gap'))

In [10]:
# look at autocorrelation of gap data from different walkers
plt.figure()
plt.title('autocorrelation of gap')
for n in range(50):
    series = pd.Series(converged_samples[:,n].sel(parameter='gap'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)
plt.savefig(SAVEPATH + '/auto_correlation_of_gap.png')

In [221]:
# look at trace of gap from different walkers
plt.figure()
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='gap'))

In [130]:
# look at autocorrelation of theta data from different walkers
plt.figure()
for n in range(50):
    series = pd.Series(converged_samples[:,n].sel(parameter='theta'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)

In [171]:
# look at trace of theta from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[:,n].sel(parameter='theta'))

In [185]:
# look at trace of n_1 from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[:,n].sel(parameter='n_1'))

In [103]:
# We can use the following notation to cycle throught the different parameter labels
for parameter in converged_samples.coords['parameter'].data:
    print(parameter)

n_1
r_1
x_g
gap
phi
theta
y_g
z_g
n_2
r_2
alpha


In [11]:
# Plot autocorrelation for all the different parameters
# set up plotting
number_columns = int(np.ceil(len(converged_samples.coords['parameter'].data)/3))
fig,axes = plt.subplots(3,number_columns)
m = 1
# go through analysis
for parameter_name in converged_samples.coords['parameter'].data:
    for n in range(50):
        series = pd.Series(converged_samples[:,n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
        plt.subplot(3,number_columns,m)
        plt.plot(autocorr_as_function_of_time)
    # label plot
    row = int(np.floor((m-1)/4))
    column = int(m-4*row)
    axes[row,column-1].set_title(parameter_name)
    fig.supxlabel('Chain Number')
    fig.supylabel('Autocorrelation')
    # increment number tracker
    m = m + 1
plt.savefig(SAVEPATH + '/all_autocorrelation.png')

In [12]:
# find the correlation time for each of these parameters (ie when autocorrelation drops to 0 for the first time)
all_parameter_indices = []
for parameter_name in converged_samples.coords['parameter'].data:
    all_walker_indices = []
    for n in range(50):
        series = pd.Series(converged_samples[:,n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
            if autocorr < 0:
                index = i
                all_walker_indices.append(index)
                break
            elif i == (len(series)-1):
                index = i
                all_walker_indices.append(index)
                print("sample " + str(n) + " of " +  parameter_name + " remains correlated")
    all_parameter_indices.append(all_walker_indices)
print(all_parameter_indices)
# Q: should I find max correlation time or mean correlation time for each parameter?
# start with easiest which is just overall max
overall_correlation_time = np.max(all_parameter_indices)
print(overall_correlation_time)

[[164, 666, 370, 511, 384, 442, 526, 260, 668, 479, 475, 552, 273, 541, 450, 342, 331, 279, 527, 539, 496, 456, 482, 450, 337, 315, 417, 388, 488, 542, 396, 422, 161, 346, 115, 316, 571, 769, 611, 220, 281, 188, 368, 795, 474, 314, 577, 136, 332, 440], [526, 850, 688, 568, 481, 735, 881, 450, 763, 538, 902, 559, 751, 527, 532, 767, 830, 850, 456, 732, 490, 507, 434, 594, 844, 472, 541, 877, 767, 964, 446, 543, 703, 585, 555, 499, 790, 455, 985, 663, 903, 450, 878, 470, 481, 892, 928, 928, 856, 457], [290, 297, 409, 201, 341, 127, 417, 345, 403, 328, 115, 117, 269, 334, 135, 89, 367, 361, 278, 303, 263, 205, 255, 382, 165, 145, 332, 348, 101, 246, 202, 269, 362, 319, 419, 125, 367, 374, 360, 325, 275, 334, 127, 67, 108, 191, 105, 66, 43, 272], [515, 375, 318, 339, 350, 364, 436, 325, 317, 265, 383, 741, 324, 260, 661, 565, 291, 290, 752, 344, 352, 697, 311, 305, 363, 325, 571, 290, 336, 337, 352, 246, 866, 367, 688, 352, 348, 581, 231, 249, 288, 346, 338, 388, 353, 402, 292, 308, 673, 3

In [13]:
# other approach where we find the mean and then take the max
mean_parameter_corr_time = np.mean(all_parameter_indices, axis=1)
max_of_mean_parameter_corr_time = int(np.max(mean_parameter_corr_time))
print(max_of_mean_parameter_corr_time)

666


In [16]:
# skip decimating for now and just look at end samples
independent_samples = converged_samples[999,:]

In [17]:
# now convert independent samples into a form that can be input into seaborn pairplot
independent_samples_pd = independent_samples.to_dataframe(name = 'independent samples')
independent_samples_pd = independent_samples_pd.reset_index()
independent_samples_pd = independent_samples_pd.pivot('chain','parameter','independent samples')
print(independent_samples_pd)

parameter     alpha       gap       n_1       n_2       phi       r_1  \
chain                                                                   
0          0.640531  0.114760  1.584504  1.598988  6.339122  0.666665   
1          0.642027  0.115683  1.583991  1.599015  6.338160  0.666192   
2          0.642058  0.112035  1.584397  1.599384  6.338590  0.666399   
3          0.638875  0.116515  1.584075  1.598907  6.336507  0.666215   
4          0.641115  0.111288  1.584215  1.599286  6.336461  0.666707   
5          0.639996  0.117541  1.584235  1.598977  6.337765  0.666580   
6          0.640102  0.115864  1.584107  1.599710  6.336280  0.666214   
7          0.640267  0.119411  1.584052  1.599649  6.336474  0.666978   
8          0.640898  0.114010  1.583712  1.599362  6.341837  0.666832   
9          0.641963  0.113685  1.584004  1.599018  6.341250  0.666289   
10         0.640794  0.112591  1.583845  1.599298  6.337461  0.666322   
11         0.640901  0.118669  1.584056  1.599503  

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_88819/3767963303.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  independent_samples_pd = independent_samples_pd.pivot('chain','parameter','independent samples')


In [18]:
full_pair_plot = sns.pairplot(independent_samples_pd)
full_pair_plot.savefig(SAVEPATH + '/pair_plot_of_end_of_run_samples')